# 短期记忆

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from rich import print as rprint
from langchain.agents.middleware import SummarizationMiddleware

from numpy import extract

load_dotenv(override=True)
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
ZHIPU_BASE_URL = os.getenv("ZHIPU_BASE_URL")

model=init_chat_model(
    model="glm-5.2",  # 模型名称
    model_provider="openai",
    api_key=ZHIPU_API_KEY,
    base_url=ZHIPU_BASE_URL,  # ZHIPU API 的基础 URL
    profile={"max_input_tokens": 1000000}
)

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[]
)

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]
})
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]
})
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。👋

请问今天有什么我可以帮你的吗？无论是解答问题、写文章、写代码，还是单纯聊聊天，我都可以协助你！

第二轮对话：
Agent: 抱歉，你还没有告诉我你的名字，所以我无法知道你叫什么。

作为一个人工智能，我也没有获取你个人信息的能力。如果你愿意的话，可以随时告诉我你想让我怎么称呼你！


In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()  #1、创建了内存级的记忆存储


agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer  #2、让agent具备了存储的能力
)

# 3、同一个thread_id共享记忆的。
config = {
    "configurable" : {
        "thread_id" : "1"
    }
}

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]},
    config=config   # 4、传入invoke()当中
)
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config
)
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。请问今天有什么我可以帮你的吗？无论是解答问题、写文章、写代码，还是随便聊聊天，我都可以为你效劳。

第二轮对话：
Agent: 你叫张三呀！有什么我可以帮你的吗？
